# OpenInfer Stage 3: vLLM on Colab T4

Run cells from top to bottom in a Colab T4 runtime. This notebook starts vLLM on localhost, calls Qwen directly, then runs the repository gateway in the same Colab session and calls it through the OpenAI SDK. The last cell downloads a JSON integration record. Do not save notebook outputs containing credentials.

## 1. Install vLLM

Select **Runtime → Change runtime type → T4 GPU** first. Installation can take several minutes. If Colab asks for a runtime restart after installation, restart and continue from the next cell.

In [ ]:
%pip -q install vllm

In [ ]:
import platform
import torch
import vllm

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print('Python:', platform.python_version())
print('GPU:', torch.cuda.get_device_name(0), 'compute capability:', torch.cuda.get_device_capability(0))
print('PyTorch:', torch.__version__, '| vLLM:', vllm.__version__)

## 2. Upload the current gateway source

In VS Code, run `python scripts/package_colab.py` from the repository root. Upload the generated `openinfer_stage3_bundle.zip` when prompted below. This includes your current local gateway code, including uncommitted changes.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import os

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload one OpenInfer bundle ZIP file')
uploaded_name, bundle_bytes = next(iter(uploaded.items()))
project_root = Path('/content/openinfer')
project_root.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(bundle_bytes)) as bundle:
    expected = {'requirements.txt', 'gateway/__init__.py', 'gateway/app/main.py', 'gateway/app/backends.py'}
    assert expected.issubset(set(bundle.namelist())), 'Bundle is missing gateway files'
    bundle.extractall(project_root)
os.chdir(project_root)
print('Gateway source ready from', uploaded_name, 'at', project_root)

In [ ]:
%pip -q install -r /content/openinfer/requirements.txt

## 3. Start the vLLM backend

The model is the same Qwen model used in Stage 1. vLLM listens only on Colab localhost. A generated backend credential is passed through the environment and is not printed.

In [ ]:
import secrets
import subprocess
import sys
import time
import httpx

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
PUBLIC_MODEL_ID = 'openinfer-qwen'
VLLM_PORT = 8001
GATEWAY_PORT = 8000
vllm_key = secrets.token_urlsafe(32)
public_key = secrets.token_urlsafe(32)
prompt = 'In one sentence, explain what prefill does in an autoregressive transformer.'

vllm_log = project_root / 'vllm.log'
vllm_env = os.environ.copy()
vllm_env['VLLM_API_KEY'] = vllm_key
vllm_command = [
    'vllm', 'serve', MODEL_ID, '--host', '127.0.0.1', '--port', str(VLLM_PORT),
    '--dtype', 'half', '--max-model-len', '2048', '--gpu-memory-utilization', '0.80',
]
vllm_log_handle = vllm_log.open('w')
vllm_process = subprocess.Popen(
    vllm_command, stdout=vllm_log_handle, stderr=subprocess.STDOUT, env=vllm_env,
    cwd=project_root,
)

models_url = f'http://127.0.0.1:{VLLM_PORT}/v1/models'
for attempt in range(60):
    if vllm_process.poll() is not None:
        raise RuntimeError('vLLM exited early. Recent log: \n' + vllm_log.read_text()[-4000:])
    try:
        ready = httpx.get(models_url, headers={'Authorization': f'Bearer {vllm_key}'}, timeout=5)
        if ready.status_code == 200 and any(item.get('id') == MODEL_ID for item in ready.json().get('data', [])):
            break
    except (httpx.HTTPError, ValueError):
        pass
    if attempt % 3 == 0:
        print(f'Waiting for vLLM: {attempt * 10}s')
    time.sleep(10)
else:
    raise TimeoutError('vLLM did not become ready. Recent log: \n' + vllm_log.read_text()[-4000:])
print('vLLM ready with model:', MODEL_ID)

## 4. Measure one direct vLLM call

This is a single non-streaming request. The duration includes local HTTP and generation; it is not TTFT or steady-state throughput.

In [ ]:
from openai import OpenAI

direct_client = OpenAI(api_key=vllm_key, base_url=f'http://127.0.0.1:{VLLM_PORT}/v1', timeout=120)
started = time.perf_counter()
direct_response = direct_client.chat.completions.create(
    model=MODEL_ID, messages=[{'role': 'user', 'content': prompt}],
    max_tokens=64, temperature=0,
)
direct_seconds = time.perf_counter() - started
print('Direct vLLM:', round(direct_seconds, 3), 'seconds')
print(direct_response.choices[0].message.content)

## 5. Start the gateway against the real backend

The gateway uses its own public API key. It forwards requests to the localhost vLLM server with the separate backend key.

In [ ]:
gateway_env = os.environ.copy()
gateway_env.update({
    'BACKEND_MODE': 'vllm',
    'BACKEND_BASE_URL': f'http://127.0.0.1:{VLLM_PORT}/v1',
    'BACKEND_API_KEY': vllm_key,
    'PUBLIC_API_KEYS': public_key,
    'PUBLIC_MODEL_ID': PUBLIC_MODEL_ID,
    'BACKEND_MODEL_ID': MODEL_ID,
})
gateway_log = project_root / 'gateway.log'
gateway_log_handle = gateway_log.open('w')
gateway_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'gateway.app.main:app', '--host', '127.0.0.1', '--port', str(GATEWAY_PORT)],
    stdout=gateway_log_handle, stderr=subprocess.STDOUT, env=gateway_env, cwd=project_root,
)
ready_url = f'http://127.0.0.1:{GATEWAY_PORT}/health/ready'
for attempt in range(30):
    if gateway_process.poll() is not None:
        raise RuntimeError('Gateway exited early. Recent log: \n' + gateway_log.read_text()[-4000:])
    try:
        gateway_ready = httpx.get(ready_url, timeout=5)
        if gateway_ready.status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(2)
else:
    raise TimeoutError('Gateway did not become ready. Recent log: \n' + gateway_log.read_text()[-4000:])
print('Gateway ready:', gateway_ready.json())

In [ ]:
gateway_client = OpenAI(api_key=public_key, base_url=f'http://127.0.0.1:{GATEWAY_PORT}/v1', timeout=120)
assert any(model.id == PUBLIC_MODEL_ID for model in gateway_client.models.list().data)
started = time.perf_counter()
gateway_raw = gateway_client.chat.completions.with_raw_response.create(
    model=PUBLIC_MODEL_ID, messages=[{'role': 'user', 'content': prompt}],
    max_tokens=64, temperature=0,
)
gateway_seconds = time.perf_counter() - started
gateway_response = gateway_raw.parse()
gateway_request_id = gateway_raw.headers.get('x-request-id')
print('Gateway to vLLM:', round(gateway_seconds, 3), 'seconds, request ID:', gateway_request_id)
print(gateway_response.choices[0].message.content)

## 6. Save the integration record

The direct and gateway calls are separate generations. Their difference is not an exact measure of network or gateway overhead. Both calls are non-streaming, so TTFT and TPOT are not measured here.

In [ ]:
import json
from datetime import datetime, timezone

gpu_memory = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits'],
    text=True,
).strip()
record = {
    'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'public_model_id': PUBLIC_MODEL_ID,
    'prompt': prompt,
    'max_tokens': 64,
    'temperature': 0,
    'environment': {
        'python': platform.python_version(), 'torch': str(torch.__version__),
        'vllm': vllm.__version__, 'gpu': torch.cuda.get_device_name(0),
        'gpu_memory_used_total_mib': gpu_memory,
    },
    'direct_vllm': {
        'elapsed_seconds': direct_seconds,
        'completion_tokens': direct_response.usage.completion_tokens if direct_response.usage else None,
        'content': direct_response.choices[0].message.content,
    },
    'gateway_to_vllm': {
        'elapsed_seconds': gateway_seconds,
        'request_id': gateway_request_id,
        'completion_tokens': gateway_response.usage.completion_tokens if gateway_response.usage else None,
        'content': gateway_response.choices[0].message.content,
    },
    'gateway_metrics': httpx.get(f'http://127.0.0.1:{GATEWAY_PORT}/metrics', timeout=5).json(),
}
output_path = project_root / 'openinfer_stage3_integration.json'
output_path.write_text(json.dumps(record, indent=2) + '\n', encoding='utf-8')
files.download(str(output_path))
print('Saved', output_path.name)

## Optional remote connection

The local integration above is the required first run. A Codespaces → Cloudflare Tunnel → Colab demonstration is optional and must comply with the current Colab runtime terms. Never tunnel directly to vLLM: its `--api-key` does not secure every endpoint. The repository contains an authenticated `gateway.colab_bridge` that exposes only `/v1/models` and `/v1/chat/completions`. Follow `STAGE3.md` only when remote access is permitted. Cloudflare Quick Tunnels are temporary and do not support SSE.

In [ ]:
# Optional: run only if remote access is permitted for this runtime.
bridge_key = secrets.token_urlsafe(32)
bridge_env = os.environ.copy()
bridge_env.update({
    'COLAB_BRIDGE_KEY': bridge_key,
    'VLLM_API_KEY': vllm_key,
    'VLLM_BASE_URL': f'http://127.0.0.1:{VLLM_PORT}/v1',
})
bridge_log = project_root / 'bridge.log'
bridge_log_handle = bridge_log.open('w')
bridge_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'gateway.colab_bridge:app', '--host', '127.0.0.1', '--port', '8002'],
    stdout=bridge_log_handle, stderr=subprocess.STDOUT, env=bridge_env, cwd=project_root,
)
for attempt in range(20):
    if bridge_process.poll() is not None:
        raise RuntimeError('Bridge exited early. Recent log: \n' + bridge_log.read_text()[-4000:])
    try:
        bridge_response = httpx.get(
            'http://127.0.0.1:8002/v1/models',
            headers={'Authorization': f'Bearer {bridge_key}'}, timeout=5,
        )
        if bridge_response.status_code == 200:
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise TimeoutError('Bridge did not become ready')
print('Bridge ready on Colab localhost:8002. Keep bridge_key private.')

In [ ]:
# Optional: create a temporary Cloudflare Quick Tunnel to the restricted bridge.
import re
import urllib.request

cloudflared_binary = Path('/tmp/cloudflared')
if not cloudflared_binary.exists():
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        cloudflared_binary,
    )
    cloudflared_binary.chmod(0o755)
tunnel_log = project_root / 'cloudflared.log'
tunnel_log_handle = tunnel_log.open('w')
tunnel_process = subprocess.Popen(
    [str(cloudflared_binary), 'tunnel', '--url', 'http://127.0.0.1:8002'],
    stdout=tunnel_log_handle, stderr=subprocess.STDOUT, cwd=project_root,
)
tunnel_url = None
for attempt in range(60):
    if tunnel_process.poll() is not None:
        raise RuntimeError('Tunnel exited early. Recent log: \n' + tunnel_log.read_text()[-4000:])
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)
assert tunnel_url, 'Tunnel URL was not reported; inspect cloudflared.log'
print('Tunnel URL:', tunnel_url)
print('Set BACKEND_BASE_URL to', tunnel_url + '/v1', 'in Codespaces.')
print('Set BACKEND_API_KEY to the private bridge_key value. Do not commit it.')

If the tunnel is permitted and running, configure the VS Code gateway using the URL above and the `bridge_key` generated in the previous cell. The tunnel URL changes each session. To stop this demo, call `terminate()` on `tunnel_process`, `bridge_process`, `gateway_process`, and `vllm_process`, or stop the Colab runtime.

## 7. Streaming metrics on Colab localhost

Run this after the vLLM setup in the same Colab session. It sends two warm-up requests, then five measured requests for each short/long prompt and 32/128-token output limit. It saves TTFT, a client-observed TPOT estimate, total latency, token counts, and GPU memory before/after each request. The downloaded JSON has no credentials. This does not use the tunnel or gateway. If you already have an older copy open in Colab, copy the code from `notebooks/colab_stream_metrics_cell.py` into a new cell there; local notebook edits do not appear automatically in the open Colab tab.

In [ ]:
"""Paste this entire file into one cell after Stage 3 starts vLLM in Colab.

It uses MODEL_ID, VLLM_PORT, and vllm_key from the Stage 3 notebook.
The saved result contains prompts and timings, never API keys.
"""

import json
import math
import platform
import random
import statistics
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import vllm
from google.colab import files
from openai import OpenAI


assert all(name in globals() for name in ("MODEL_ID", "VLLM_PORT", "vllm_key")), (
    "Run the Stage 3 vLLM setup cells first, in this Colab session."
)
metrics_client = OpenAI(
    api_key=vllm_key,
    base_url=f"http://127.0.0.1:{VLLM_PORT}/v1",
    timeout=120,
    max_retries=0,
)
assert any(item.id == MODEL_ID for item in metrics_client.models.list().data), (
    "vLLM is not ready. Restart it using the Stage 3 notebook."
)


def gpu_memory_mib():
    line = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).splitlines()[0]
    used, total = line.split(",")[:2]
    return {"used_mib": int(used.strip()), "total_mib": int(total.strip())}


def make_prompt(bucket, trial):
    opener = (
        f"Trial {trial}. Explain how a transformer uses a KV cache during "
        "autoregressive inference. Give a detailed technical explanation."
    )
    if bucket == "short":
        return opener
    words = ["attention", "prefill", "decode", "memory", "tokens", "layers", "latency", "cache"]
    rng = random.Random(20261017 + trial)
    context = " ".join(rng.choices(words, k=450))
    return f"Trial {trial}. Context to read: {context}\n{opener}"


def one_stream(bucket, output_limit, trial):
    prompt = make_prompt(bucket, trial)
    memory_before = gpu_memory_mib()
    started = time.perf_counter()
    first_text_at = None
    last_text_at = None
    content_chunks = 0
    usage = None
    finish_reason = None
    stream = metrics_client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=output_limit,
        temperature=0,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={"ignore_eos": True},
    )
    try:
        for chunk in stream:
            arrived = time.perf_counter()
            if chunk.usage is not None:
                usage = chunk.usage
            if chunk.choices:
                choice = chunk.choices[0]
                if choice.finish_reason:
                    finish_reason = choice.finish_reason
                if choice.delta.content:
                    if first_text_at is None:
                        first_text_at = arrived
                    last_text_at = arrived
                    content_chunks += 1
    finally:
        stream.close()
    ended = time.perf_counter()
    memory_after = gpu_memory_mib()
    completion_tokens = usage.completion_tokens if usage else None
    ttft = first_text_at - started if first_text_at is not None else None
    tpot = (
        (last_text_at - first_text_at) / (completion_tokens - 1)
        if first_text_at is not None and last_text_at is not None
        and completion_tokens is not None and completion_tokens > 1 and content_chunks > 1
        else None
    )
    return {
        "bucket": bucket,
        "trial": trial,
        "prompt": prompt,
        "prompt_tokens": usage.prompt_tokens if usage else None,
        "output_limit": output_limit,
        "completion_tokens": completion_tokens,
        "finish_reason": finish_reason,
        "content_chunks": content_chunks,
        "ttft_seconds": ttft,
        "tpot_estimate_seconds": tpot,
        "elapsed_seconds": ended - started,
        "output_tokens_per_second": completion_tokens / (ended - started) if completion_tokens else None,
        "gpu_memory_before": memory_before,
        "gpu_memory_after": memory_after,
    }


def nearest_rank_p95(values):
    values = sorted(value for value in values if value is not None)
    return values[max(0, math.ceil(0.95 * len(values)) - 1)] if values else None


# Warm-up calls are excluded. A different prompt is used for each measured trial.
for warmup_trial in (-2, -1):
    one_stream("short", 32, warmup_trial)

raw = []
for bucket in ("short", "long"):
    for output_limit in (32, 128):
        for trial in range(5):
            # Keep prompts distinct across conditions to reduce prefix-cache reuse.
            prompt_trial = trial + (100 if bucket == "long" else 0) + (1000 if output_limit == 128 else 0)
            sample = one_stream(bucket, output_limit, prompt_trial)
            raw.append(sample)
            print(
                bucket, output_limit, f"trial={trial + 1}/5",
                f"input={sample['prompt_tokens']}",
                f"output={sample['completion_tokens']}",
                f"TTFT={sample['ttft_seconds']:.3f}s" if sample["ttft_seconds"] is not None else "TTFT unavailable",
                f"elapsed={sample['elapsed_seconds']:.3f}s",
                flush=True,
            )

summary = []
for bucket in ("short", "long"):
    for output_limit in (32, 128):
        group = [item for item in raw if item["bucket"] == bucket and item["output_limit"] == output_limit]
        metric = lambda key: [item[key] for item in group if item[key] is not None]
        summary.append({
            "bucket": bucket,
            "output_limit": output_limit,
            "samples": len(group),
            "median_prompt_tokens": statistics.median(metric("prompt_tokens")),
            "median_completion_tokens": statistics.median(metric("completion_tokens")),
            "median_ttft_seconds": statistics.median(metric("ttft_seconds")) if metric("ttft_seconds") else None,
            "p95_ttft_seconds": nearest_rank_p95(metric("ttft_seconds")),
            "median_tpot_estimate_seconds": (
                statistics.median(metric("tpot_estimate_seconds"))
                if metric("tpot_estimate_seconds") else None
            ),
            "median_elapsed_seconds": statistics.median(metric("elapsed_seconds")),
            "p95_elapsed_seconds": nearest_rank_p95(metric("elapsed_seconds")),
            "median_output_tokens_per_second": (
                statistics.median(metric("output_tokens_per_second"))
                if metric("output_tokens_per_second") else None
            ),
        })

record = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "environment": {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "vllm": vllm.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "vllm_dtype_from_stage3_command": "half",
        "vllm_max_model_len_from_stage3_command": 2048,
    },
    "method": {
        "path": "Colab Python client -> Colab localhost vLLM",
        "stream": True,
        "temperature": 0,
        "ignore_eos": True,
        "warmup_requests": 2,
        "repeats_per_condition": 5,
        "concurrency": 1,
        "ttft_definition": "client request start to first nonempty streamed text delta",
        "tpot_estimate_definition": "first to last text delta divided by completion_tokens minus one; chunking can make this approximate",
        "gpu_memory_definition": "nvidia-smi whole-GPU memory.used before and after each request, not peak or per-request KV memory",
    },
    "summary": summary,
    "raw": raw,
}
output_path = Path("/content/openinfer_stream_metrics.json")
output_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
print("Saved", output_path)
for row in summary:
    print(row)
files.download(str(output_path))


## 8. Direct vLLM concurrency on Colab localhost

This uses the same 1/4/8/16-client non-streaming workload as the remote benchmark, but calls vLLM directly on Colab localhost. Run it after the vLLM setup in the same session. It makes two excluded warm-up calls, then 16 requests per level, saves raw latency and token counts, and downloads `openinfer_direct_concurrency.json`. For an already-open Colab tab, copy the contents of `notebooks/colab_direct_concurrency_cell.py` into a new code cell. The whole-GPU memory readings before/after each batch do not measure peak or per-request KV-cache use.

In [ ]:
"""Paste into a new cell in the Stage 3 Colab notebook after vLLM is running.

This measures the direct localhost vLLM path with the same workload used by
scripts/benchmark_gateway.py. No credentials are written to the result.
"""

import asyncio
import json
import math
import platform
import subprocess
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import httpx
import torch
import vllm
from google.colab import files


assert all(name in globals() for name in ("MODEL_ID", "VLLM_PORT", "vllm_key")), (
    "Run the Stage 3 vLLM setup cells first, in this Colab session."
)
DIRECT_LEVELS = (1, 4, 8, 16)
DIRECT_REQUESTS_PER_LEVEL = 16
DIRECT_MAX_TOKENS = 32
DIRECT_PROMPT_TEMPLATE = (
    "Explain one practical use of a transformer KV cache in one short sentence. "
    "Request number {number}."
)


def direct_gpu_memory_mib():
    line = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        text=True,
    ).splitlines()[0]
    used, total = line.split(",")[:2]
    return {"used_mib": int(used.strip()), "total_mib": int(total.strip())}


def direct_p95(values):
    values = sorted(values)
    return values[max(0, math.ceil(0.95 * len(values)) - 1)] if values else None


async def direct_level(client, concurrency, request_count):
    gate = asyncio.Semaphore(concurrency)

    async def one_request(number):
        payload = {
            "model": MODEL_ID,
            "messages": [{"role": "user", "content": DIRECT_PROMPT_TEMPLATE.format(number=number)}],
            "max_tokens": DIRECT_MAX_TOKENS,
            "temperature": 0,
            "stream": False,
        }
        async with gate:
            started = time.perf_counter()
            try:
                response = await client.post(
                    f"http://127.0.0.1:{VLLM_PORT}/v1/chat/completions", json=payload
                )
                elapsed = time.perf_counter() - started
                try:
                    body = response.json()
                except ValueError:
                    body = {}
                usage = body.get("usage") if isinstance(body, dict) else None
                return {
                    "request_number": number,
                    "status_code": response.status_code,
                    "elapsed_seconds": elapsed,
                    "prompt_tokens": usage.get("prompt_tokens") if isinstance(usage, dict) else None,
                    "completion_tokens": usage.get("completion_tokens") if isinstance(usage, dict) else None,
                }
            except httpx.HTTPError as exc:
                return {
                    "request_number": number,
                    "status_code": None,
                    "elapsed_seconds": time.perf_counter() - started,
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "client_error": type(exc).__name__,
                }

    memory_before = direct_gpu_memory_mib()
    batch_started = time.perf_counter()
    requests = await asyncio.gather(*(one_request(i) for i in range(request_count)))
    batch_seconds = time.perf_counter() - batch_started
    memory_after = direct_gpu_memory_mib()
    success = [item for item in requests if item["status_code"] == 200]
    latencies = [item["elapsed_seconds"] for item in success]
    output_tokens = sum(item["completion_tokens"] or 0 for item in success)
    return {
        "concurrency": concurrency,
        "request_count": request_count,
        "successful_requests": len(success),
        "status_counts": dict(Counter(str(item["status_code"]) for item in requests)),
        "batch_seconds": batch_seconds,
        "requests_per_second": len(success) / batch_seconds,
        "output_tokens_per_second": output_tokens / batch_seconds,
        "latency_p50_seconds": (
            sorted(latencies)[max(0, math.ceil(0.5 * len(latencies)) - 1)]
            if latencies else None
        ),
        "latency_p95_seconds": direct_p95(latencies),
        "gpu_memory_before": memory_before,
        "gpu_memory_after": memory_after,
        "raw_requests": requests,
    }


async def direct_benchmark():
    timeout = httpx.Timeout(130.0, connect=10.0)
    limits = httpx.Limits(max_connections=20)
    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {vllm_key}"},
        timeout=timeout,
        limits=limits,
    ) as client:
        models = await client.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models")
        models.raise_for_status()
        assert MODEL_ID in {item["id"] for item in models.json()["data"]}, "vLLM model unavailable"
        for warmup in range(2):
            result = await direct_level(client, 1, 1)
            if result["successful_requests"] != 1:
                raise RuntimeError(f"Warm-up {warmup + 1} failed: {result['status_counts']}")
        levels = []
        for concurrency in DIRECT_LEVELS:
            result = await direct_level(client, concurrency, DIRECT_REQUESTS_PER_LEVEL)
            levels.append(result)
            print(
                f"concurrency={concurrency}: {result['successful_requests']}/16 succeeded, "
                f"{result['requests_per_second']:.2f} requests/s, "
                f"p95={result['latency_p95_seconds']}s, "
                f"statuses={result['status_counts']}",
                flush=True,
            )
    return levels


# Colab has an event loop already. Run the async workload in a separate thread.
with ThreadPoolExecutor(max_workers=1) as pool:
    direct_levels = pool.submit(lambda: asyncio.run(direct_benchmark())).result()

direct_record = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "environment": {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "vllm": vllm.__version__,
        "gpu": torch.cuda.get_device_name(0),
    },
    "method": {
        "path": "Colab Python client -> Colab localhost vLLM",
        "prompt_template": DIRECT_PROMPT_TEMPLATE,
        "max_tokens": DIRECT_MAX_TOKENS,
        "temperature": 0,
        "stream": False,
        "warmup_requests": 2,
        "requests_per_level": DIRECT_REQUESTS_PER_LEVEL,
        "levels": list(DIRECT_LEVELS),
        "gpu_memory_definition": "nvidia-smi whole-GPU memory.used before and after each batch, not peak or per-request KV memory",
    },
    "levels": direct_levels,
}
direct_output = Path("/content/openinfer_direct_concurrency.json")
direct_output.write_text(json.dumps(direct_record, indent=2) + "\n", encoding="utf-8")
print("Saved", direct_output)
files.download(str(direct_output))
